In [55]:
import pandas as pd
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')

In [56]:
train.head(20)

,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3,24,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5,70,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5,26,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5,34,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8,26,Male,Child,Single/Never Married,Primary education,Informally employed
5,Kenya,2018,uniqueid_6,No,Rural,No,7,26,Female,Spouse,Married/Living together,Primary education,Informally employed
6,Kenya,2018,uniqueid_7,No,Rural,Yes,7,32,Female,Spouse,Married/Living together,Primary education,Self employed
7,Kenya,2018,uniqueid_8,No,Rural,Yes,1,42,Female,Head of Household,Married/Living together,Tertiary education,Formally employed Government
8,Kenya,2018,uniqueid_9,Yes,Rural,Yes,3,54,Male,Head of Household,Married/Living together,Secondary education,Farming and Fishing
9,Kenya,2018,uniqueid_10,No,Urban,Yes,3,76,Female,Head of Household,Divorced/Seperated,No formal education,Remittance Dependent


In [57]:
train.shape


(23524, 13)

In [58]:
train.columns

Index(['country', 'year', 'uniqueid', 'bank_account', 'location_type',
       'cellphone_access', 'household_size', 'age_of_respondent',
       'gender_of_respondent', 'relationship_with_head', 'marital_status',
       'education_level', 'job_type'],
      dtype='object')

In [59]:
train["bank_account"].value_counts()

bank_account
No     20212
Yes     3312
Name: count, dtype: int64

In [60]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   country                 23524 non-null  object
 1   year                    23524 non-null  int64 
 2   uniqueid                23524 non-null  object
 3   bank_account            23524 non-null  object
 4   location_type           23524 non-null  object
 5   cellphone_access        23524 non-null  object
 6   household_size          23524 non-null  int64 
 7   age_of_respondent       23524 non-null  int64 
 8   gender_of_respondent    23524 non-null  object
 9   relationship_with_head  23524 non-null  object
 10  marital_status          23524 non-null  object
 11  education_level         23524 non-null  object
 12  job_type                23524 non-null  object
dtypes: int64(3), object(10)
memory usage: 2.3+ MB


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler


#convert the label to numerical data

le=LabelEncoder()
train["bank_account"]=le.fit_transform(train["bank_account"])

#separate training features from target variable
X=train.drop("bank_account",axis=1)
y=train["bank_account"]



0        1
1        0
2        1
3        0
4        0
        ..
23519    0
23520    0
23521    0
23522    0
23523    0
Name: bank_account, Length: 23524, dtype: int64


In [62]:
def preprocessing_data(data):
    #convert numerical data from integer to float

    float_Array = data[["household_size", "age_of_respondent", "year"]].values.astype(float)

    #conversion of categorical features to numerical data using OneHotEncoder

    categ = ["relationship_with_head",
             "marital_status",
             "education_level",
             "job_type",
             "country"]

    data = pd.get_dummies(data,prefix_sep="_",columns=categ)

    #label Encoder conversion
    data['location_type'] = le.fit_transform(data['location_type'])
    data['cellphone_access'] = le.fit_transform(data['cellphone_access'])
    data['gender_of_respondent'] = le.fit_transform(data['gender_of_respondent'])

    #remove uniqueId
    data = data.drop("uniqueid",axis=1)

    # # scale our data into range of 0 and 1
    scaler = MinMaxScaler(feature_range=(0,1))
    data = scaler.fit_transform(data)


    return data


In [63]:
preprocessed_train = preprocessing_data(X)
preprocessed_test = preprocessing_data(test)

In [64]:
print(preprocessed_train[:1])

[[1.        0.        1.        0.1       0.0952381 0.        0.
  0.        0.        0.        0.        1.        0.        0.
  1.        0.        0.        0.        0.        0.        1.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        1.        1.        0.
  0.        0.       ]]


In [65]:
# shape of the processed train set
print(preprocessed_train.shape)

(23524, 37)


In [85]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(preprocessed_train, y, test_size=0.2, random_state=42, stratify=y)

In [86]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [87]:
y_pred = model.predict(X_test)

In [88]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test,y_pred)

print("Mean Absolute Error:", mae)


Mean Absolute Error: 0.11115834218916047


In [97]:
# Réponses connues (Train) et prédites (Test), indexées par identifiant
train_ids = train["uniqueid"] + " x " + train["country"]
test_ids = test["uniqueid"] + " x " + test["country"]

answers = pd.concat([
    pd.Series(train["bank_account"].values, index=train_ids),
    pd.Series(test_pred, index=test_ids),
])

# Remplir le sample dans son ordre exact
full = sample[[id_col]].copy()
full[target_col] = full[id_col].map(answers)

# Vérifications
print(full.shape)                     # doit être (33610, 2)
print(full[target_col].isna().sum())  # doit être 0

full[target_col] = full[target_col].astype(int)
full.to_csv("first_submission.csv", index=False)

(33610, 2)
0


In [98]:
from sklearn.metrics import mean_absolute_error

# 1. Le modèle fonctionne-t-il ? (sur tes 10 % de validation)
print("MAE validation :", mean_absolute_error(y_test, model.predict(X_test)))

# 2. Que prédit-il sur Test.csv ?
print("Part de 1 prédits sur Test.csv :", test_pred.mean())

# 3. Le fichier contient-il bien ces prédictions ?
check = pd.read_csv("first_submission.csv")
ids_test = test["uniqueid"] + " x " + test["country"]
in_file = check.set_index("unique_id").loc[ids_test, "bank_account"].values
print("Fichier = prédictions du modèle :", (in_file == test_pred).all())

MAE validation : 0.11115834218916047
Part de 1 prédits sur Test.csv : 0.0709894903827087
Fichier = prédictions du modèle : True


In [100]:
submission.to_csv("test_only_submission.csv", index=False)
print(pd.read_csv("test_only_submission.csv").shape)  # doit être (10086, 2)

(10086, 2)
